In [12]:
from google.colab import files
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
%matplotlib inline

uploaded = files.upload()
credit_filename = next(iter(uploaded))

df = pd.read_csv(io.StringIO(uploaded[credit_filename].decode('utf-8')))
display(df.head())
df.info()
df['class'].unique()

X = df.columns.drop("class")
y = df['class']
df_encoded = pd.get_dummies(df[X])
df_encoded.shape

from sklearn.model_selection import train_test_split
#splitting data into train and test datasets in 85:15 ratio
X_train,X_test,y_train,y_test = train_test_split(df_encoded, y,test_size=0.15,random_state=100)
# Checking the shapes of the resulting datasets
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)



Saving credit.csv to credit (8).csv


,over_draft,credit_usage,credit_history,purpose,current_balance,Average_Credit_Balance,employment,location,personal_status,other_parties,...,property_magnitude,cc_age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   over_draft              1000 non-null   object
 1   credit_usage            1000 non-null   int64 
 2   credit_history          1000 non-null   object
 3   purpose                 1000 non-null   object
 4   current_balance         1000 non-null   int64 
 5   Average_Credit_Balance  1000 non-null   object
 6   employment              1000 non-null   object
 7   location                1000 non-null   int64 
 8   personal_status         1000 non-null   object
 9   other_parties           1000 non-null   object
 10  residence_since         1000 non-null   int64 
 11  property_magnitude      1000 non-null   object
 12  cc_age                  1000 non-null   int64 
 13  other_payment_plans     1000 non-null   object
 14  housing                 1000 non-null   object
 15  exist

In [15]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data has been scaled.")

Data has been scaled.


In [17]:
# Re-train the Logistic Regression model with scaled data
from sklearn.linear_model import LogisticRegression
model_scaled = LogisticRegression(max_iter=1000)
model_scaled.fit(X_train_scaled, y_train)

train_accuracy_scaled = model_scaled.score(X_train_scaled, y_train)
print("Train accuracy (scaled data) = ", train_accuracy_scaled)
test_accuracy_scaled = model_scaled.score(X_test_scaled, y_test)
print("Test accuracy (scaled data) = ", test_accuracy_scaled)

train_predictions_scaled = model_scaled.predict(X_train_scaled)
test_predictions_scaled = model_scaled.predict(X_test_scaled)

from sklearn.metrics import confusion_matrix, classification_report

# Training Confusion Matrix
train_conf_matrix_scaled = confusion_matrix(y_train, train_predictions_scaled)
print("\nTraining Confusion Matrix:")
display(pd.DataFrame(train_conf_matrix_scaled, columns=model_scaled.classes_, index=model_scaled.classes_))

# Training Classification Report
print("\nTraining Classification Report:")
print(classification_report(y_train, train_predictions_scaled))

# Test Confusion Matrix
test_conf_matrix_scaled = confusion_matrix(y_test, test_predictions_scaled)
print("\nTest Confusion Matrix:")
display(pd.DataFrame(test_conf_matrix_scaled, columns=model_scaled.classes_, index=model_scaled.classes_))

# Test Classification Report
print("\nTest Classification Report:")
print(classification_report(y_test, test_predictions_scaled))

train_correct_predictions = train_conf_matrix_scaled[0][0]+train_conf_matrix_scaled[1][1]
train_total_predictions = train_conf_matrix_scaled.sum()
train_accuracy = train_correct_predictions/train_total_predictions
print('train accuracy:',train_accuracy)

test_correct_predictions = test_conf_matrix_scaled[0][0]+test_conf_matrix_scaled[1][1]
total_predictions = test_conf_matrix_scaled.sum()
test_accuracy = test_correct_predictions/total_predictions
print('test accuracy:',test_accuracy)

Train accuracy (scaled data) =  0.8
Test accuracy (scaled data) =  0.7333333333333333

Training Confusion Matrix:


,bad,good
bad,149,108
good,62,531



Training Classification Report:
              precision    recall  f1-score   support

         bad       0.71      0.58      0.64       257
        good       0.83      0.90      0.86       593

    accuracy                           0.80       850
   macro avg       0.77      0.74      0.75       850
weighted avg       0.79      0.80      0.79       850


Test Confusion Matrix:


,bad,good
bad,19,24
good,16,91



Test Classification Report:
              precision    recall  f1-score   support

         bad       0.54      0.44      0.49        43
        good       0.79      0.85      0.82       107

    accuracy                           0.73       150
   macro avg       0.67      0.65      0.65       150
weighted avg       0.72      0.73      0.72       150

train accuracy: 0.8
test accuracy: 0.7333333333333333
